# 04 模型评估

仅评估本项目真实训练产生的权重，不回退到官方预训练权重。未训练模型保留在实验表中并标记为“未训练”。最低要求是至少存在一个 YOLO11 和一个 YOLO26 权重。

In [ ]:
from pathlib import Path
from time import perf_counter
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

def locate_project_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'src').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    raise FileNotFoundError('找不到项目根目录')

PROJECT_ROOT = locate_project_root()
DATA_YAML = PROJECT_ROOT / 'data' / 'data.yaml'
SAMPLE_IMAGE = PROJECT_ROOT / 'data' / 'evaluation_sample.jpg'
MODEL_NAMES = ['YOLO11n', 'YOLO11s', 'YOLO26n', 'YOLO26s']
WEIGHTS = {name: PROJECT_ROOT / 'weights' / name / 'best.pt' for name in MODEL_NAMES}
available = {name: path for name, path in WEIGHTS.items() if path.is_file()}
if not any(name.startswith('YOLO11') for name in available):
    raise FileNotFoundError('至少需要一个真实训练得到的 YOLO11 权重')
if not any(name.startswith('YOLO26') for name in available):
    raise FileNotFoundError('至少需要一个真实训练得到的 YOLO26 权重')
if not SAMPLE_IMAGE.is_file():
    raise FileNotFoundError(f'找不到单图推理耗时样例：{SAMPLE_IMAGE}')
available

In [ ]:
rows = []
for name in MODEL_NAMES:
    weight = WEIGHTS[name]
    if not weight.is_file():
        rows.append({'model': name, 'status': '未训练', 'precision': None, 'recall': None, 'map50': None, 'map50_95': None, 'weight_mb': None, 'inference_ms': None})
        continue
    model = YOLO(str(weight))
    metrics = model.val(data=str(DATA_YAML), split='test')
    started = perf_counter()
    model.predict(source=str(SAMPLE_IMAGE), verbose=False)
    inference_ms = (perf_counter() - started) * 1000
    rows.append({'model': name, 'status': '已评估', 'precision': float(metrics.box.mp), 'recall': float(metrics.box.mr), 'map50': float(metrics.box.map50), 'map50_95': float(metrics.box.map), 'weight_mb': weight.stat().st_size / 1024 / 1024, 'inference_ms': inference_ms})
comparison = pd.DataFrame(rows)
evaluated = comparison['status'].eq('已评估')
recommended_index = comparison.loc[evaluated].sort_values(['map50_95', 'inference_ms'], ascending=[False, True]).index[0]
comparison['recommended'] = comparison.index == recommended_index
reports_dir = PROJECT_ROOT / 'reports'
reports_dir.mkdir(exist_ok=True)
comparison.to_csv(reports_dir / 'model_comparison.csv', index=False, encoding='utf-8-sig')
comparison

In [ ]:
plotted = comparison.dropna(subset=['inference_ms', 'map50_95'])
ax = plotted.plot.scatter(x='inference_ms', y='map50_95', title='模型精度与速度对比')
for _, row in plotted.iterrows():
    ax.annotate(row['model'], (row['inference_ms'], row['map50_95']))
ax.set_xlabel('单张图片推理耗时（ms）')
ax.set_ylabel('mAP50-95')
plt.tight_layout()
plt.savefig(reports_dir / 'model_comparison.png', dpi=160)